## This demo presents the implementation for the RSPY-323 story.

In [ ]:
import requests
import os
import json
import pprint
# Init environment before running a demo notebook.
from resources.utils import *

pp = pprint.PrettyPrinter(indent=2, width=80, sort_dicts=False, compact=True)
session = requests.Session()
user = os.environ["JUPYTERHUB_USER"] if cluster_mode else os.environ["RSPY_HOST_USER"]
auxip_client, cadip_client, catalog_client, staging_client, prip_client = init_demo(owner_id = user)
if os.getenv("RSPY_LOCAL_MODE") == "1":
    href = "http://rs-server-adgs:8000"
    href_staging = "http://rs-server-staging:8000"
else:
    href = os.environ["RSPY_WEBSITE"]
    href_staging = "https://dev-rspy.esa-copernicus.eu"
    session.cookies.set ("session", os.environ["RSPY_OAUTH2_COOKIE"])

adgs_collection_id = "adgs"

In [ ]:
# Init the dask cluster
from resources.dask_clusters.dask_main_env import *
await init_dask_cluster_staging()

In [ ]:
# Create a test collection 
collection = create_test_collection()

### Check the added collection with the rs-server-catalog. This collection should be empty.

In [ ]:
# Check the catalog for agrosu_my_test_collection
catalog_collection = catalog_client.get_collection(collection_id=TEST_COLLECTION, owner_id=user)
assert isinstance(catalog_collection, CollectionClient)
assert len(list(catalog_collection.get_items())) == 0
print(f"No items found in the '{TEST_COLLECTION}' collection")

### Creating a staging body to start the staging process

In [ ]:
items_collection = auxip_client.search(max_items = 10, collections = [adgs_collection_id])
assert len(items_collection) > 0

### Start the staging process for adgs

In [ ]:
staging_resp_list = []
for items in items_collection:
    staging_resp_list.append(staging_client.run_staging(items.to_dict(), TEST_COLLECTION))

started_job_id_list = []

for resp in staging_resp_list:
    staging_client.wait_for_jobs(resp, logger)

### Check the catalog for the present items.

In [ ]:
# Check the catalog for agrosu_my_test_collection
catalog_collection = catalog_client.get_collection(collection_id=TEST_COLLECTION, owner_id=user)
assert isinstance(catalog_collection, CollectionClient)
assert len(list(catalog_collection.get_items())) > 0
for item in catalog_collection.get_items():
    print(f"Item {item.id} has {len(item.assets)} assets")     

### Delete one item from the collection

In [ ]:
item_to_delete = "S1A_OPER_MPL_ORBSCT_20240115T150704_99999999T999999_0025"
result = catalog_client.remove_item(collection_id=TEST_COLLECTION, owner_id=user, item_id=item_to_delete)
assert result.json()["deleted item"] == "S1A_OPER_MPL_ORBSCT_20240115T150704_99999999T999999_0025"
pp.pprint(result.json())

### Delete the whole collection

In [ ]:
result = catalog_client.remove_collection(collection_id=TEST_COLLECTION, owner_id=user)
assert result.json()["deleted collection"] == TEST_COLLECTION
pp.pprint(result.json())